In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL
from myPythonTools.TopoAna import cube_panels as CP


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)
importlib.reload(CP)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
def ridge_scale_from_fields(fields, sigmas_km, dx_km=3.336, gamma=0.75):
    """
    fields[k] : 2-D array [y, x], already blurred to width sigmas_km[k] (km).
    Returns, per pixel, at the width where the ridge response is strongest:
      sig_best : that width (km)
      R1_best  : fair ridge strength, about 0.35 x ridge height (m)
      th_cross : cross-ridge direction, degrees from +x toward +y, in [0, 180)
    """
    ridge_crit = 0.5 
    shape    = fields[0].shape
    R_best   = np.full(shape, -np.inf)
    R1_best  = np.full(shape, np.nan)
    sig_best = np.full(shape, np.nan)
    th_cross = np.full(shape, np.nan)
    for F, s_km in zip(fields, sigmas_km):
        s = s_km / dx_km                                   # blur width in cells
        Fy, Fx = np.gradient(F)                            # slopes (axis 0 = y, axis 1 = x)
        Lyy = np.gradient(Fy, axis=0)                      # curvatures
        Lxx = np.gradient(Fx, axis=1)
        Lxy = np.gradient(Fx, axis=0)
        half_tr = 0.5 * (Lxx + Lyy)
        rad     = np.sqrt(0.25 * (Lxx - Lyy)**2 + Lxy**2)
        lam_q   = half_tr - rad                            # curvature across the crest
        lam_p   = half_tr + rad                            # curvature along the crest
        ridge_like = (lam_q < 0) & (np.abs(lam_p) < ridge_crit * np.abs(lam_q))
        R  = np.where(ridge_like, (s**2)**gamma * (-lam_q), -np.inf)   # chooses the width
        R1 = (s**2) * (-lam_q)                                          # compares strengths
        better   = R > R_best
        R_best   = np.where(better, R, R_best)
        R1_best  = np.where(better, R1, R1_best)
        sig_best = np.where(better, s_km, sig_best)
        th_p     = 0.5 * np.degrees(np.arctan2(2 * Lxy, Lxx - Lyy))
        th_cross = np.where(better, (th_p + 90.) % 180., th_cross)
    return sig_best, R1_best, th_cross

In [ ]:
from scipy.ndimage import gaussian_filter

TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/
Case = 'ne30pg3_Sco100_Rw050_Wm1_smooth_gauss_scale_v207_e05'
# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]
List=RL.rdridgelist( ListFile )


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo_smooth*nc')
topofile = foo[-1] # "/glade/work/juliob/Topo/NCARTopoJTB-fire/cases/ne30pg3_Sco100_Rw050_Wm1_smooth_gauss_scale_v207_e05/output/topo_smooth_gmted2010_modis_bedmachine_nc3000_Co060.nc"

print( ListFile)
print( topofile )
# --- files ---------------------------------------------------------------

nc=3000

topo = xr.open_dataset(topofile)

In [ ]:
print( List.keys() )
print( list(topo.variables ))

In [ ]:
sm   = np.reshape(topo['terr_sm'].values,  (6, nc, nc))
dev  = np.reshape(topo['terr_dev'].values, (6, nc, nc))



tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

gauss_scale = np.reshape( topo['gauss_scale'].values , (6,nc,nc) )
gauss_strng = np.reshape( topo['gauss_strength'].values , (6,nc,nc) )
gauss_angle = np.reshape( topo['gauss_angle'].values , (6,nc,nc) )





In [ ]:
ixs , iys, ipn = np.rint(List.xs).astype(int) -1, np.rint(List.ys).astype(int) -1,np.rint(List.MyPanel).astype(int) -1
gauss_scale_seed = gauss_scale[ ipn[:] , iys[:] , ixs[:] ]
dev_seed = dev[ ipn[:] , iys[:] , ixs[:] ]

gauss_scale_f = copy.deepcopy(gauss_scale)
gauss_scale_f[ gauss_scale_f<1. ]=100.

print( np.sum(gauss_scale_seed==0.)/len(gauss_scale_seed) )

In [ ]:
from scipy.ndimage import maximum_filter
r = 5
gr_f = np.where(gauss_strng > 0, gauss_strng, -1.0)
best = maximum_filter(gr_f, size=(1, 2*r+1, 2*r+1))       # per face, max within +/- r cells
# scale at that strongest cell: pick it by searching the window (slow but simple) for the seeds only
def snap_scale(p, y, x):
    y0, y1, x0, x1 = max(y-r,0), min(y+r+1,nc), max(x-r,0), min(x+r+1,nc)
    w = gr_f[p, y0:y1, x0:x1]; j = np.unravel_index(np.argmax(w), w.shape)
    return gauss_scale[p, y0+j[0], x0+j[1]]
s_snap = np.array([snap_scale(p, y, x) for p, y, x in zip(ipn, iys, ixs)])
ok = (gauss_scale_seed > 0) & (s_snap > 0)
print("seed scale = snapped scale:", np.mean(gauss_scale_seed[ok] == s_snap[ok]).round(3))
print("seed scale larger:", np.mean(gauss_scale_seed[ok] > s_snap[ok]).round(3))

In [ ]:
def climb_scale(p, y, x):
    s0, a = gauss_scale[p, y, x], gauss_angle[p, y, x]
    if s0 <= 0:
        return 0.
    t = np.radians(a); dx, dy = np.cos(t), np.sin(t)           # across the ridge, grid cells
    kmax = max(1, int(np.ceil(2 * s0 / 3.336)))                 # stay within ~2 sigma of the seed's own scale
    best_v, best_s = gr_f[p, y, x], s0
    for sgn in (1, -1):
        prev = gr_f[p, y, x]
        for k in range(1, kmax + 1):
            yy, xx = int(round(y + sgn*k*dy)), int(round(x + sgn*k*dx))
            if not (0 <= yy < nc and 0 <= xx < nc):
                break
            v = gr_f[p, yy, xx]
            if v < prev:                                         # stop once strength starts falling
                break
            prev = v
            if v > best_v:
                best_v, best_s = v, gauss_scale[p, yy, xx]
    return best_s

s_climb = np.array([climb_scale(p, y, x) for p, y, x in zip(ipn, iys, ixs)])
ok = (gauss_scale_seed > 0) & (s_climb > 0)
print("seed = own crest :", np.mean(gauss_scale_seed[ok] == s_climb[ok]).round(3))
print("seed larger      :", np.mean(gauss_scale_seed[ok] >  s_climb[ok]).round(3))
print("seed smaller     :", np.mean(gauss_scale_seed[ok] <  s_climb[ok]).round(3))

In [ ]:
sc = s_climb[s_climb > 0]
for c in (4, 5, 6):
    n = np.clip(np.rint(c * sc / 3.336), 4, 42)            # 42 = nsw for Sco100, ratio 1.0
    print(f"c={c}: median {np.median(n):4.0f}  10-90% {np.percentile(n,10):4.0f}-{np.percentile(n,90):4.0f}"
          f"  at max: {np.mean(n == 42):.2f}  at min: {np.mean(n == 4):.2f}")

In [ ]:
plt.scatter( gauss_scale_seed , dev_seed, s=1 )

In [ ]:
xlim=np.asarray( [0,500] ) + 1500  #-70+1000
ylim=np.asarray( [0,500] ) + 2000 # -70
fig,axs=plt.subplots(1,3, figsize=(36,8) )
ipl=1
ax=axs[0]
co = ax.contourf( gauss_scale_f[ipl,:,:] ,levels=np.linspace(0,80,num=31) , extend='both' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
#ax.scatter( ixs, iys, marker='+', color='white' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)
ax=axs[1]
co = ax.contourf( dev[ipl,:,:] ,levels=np.linspace(-1000,1000,num=31) , cmap='terrain' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
#ax.scatter( ixs, iys, marker='+', color='black' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)

ax=axs[2]
co = ax.contourf( gauss_strng[ipl,:,:] ,levels=0.5*np.linspace(-1000,1000,num=31) , cmap='terrain' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
#ax.scatter( ixs, iys, marker='+', color='black' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)



In [ ]:
xlim=np.asarray( [900,1100] ) #-70+1000
ylim=np.asarray( [2800,3000] ) # -70
fig,axs=plt.subplots(1,3, figsize=(36,8) )
ax=axs[0]
co = ax.contourf( gauss_scale_f[3,:,:] ,levels=np.linspace(0,80,num=31) , extend='both' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
ax.scatter( ixs, iys, marker='+', color='white' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)
ax=axs[1]
co = ax.contourf( dev[3,:,:] ,levels=np.linspace(-1000,1000,num=31) , cmap='terrain' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
ax.scatter( ixs, iys, marker='+', color='black' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)

ax=axs[2]
co = ax.contourf( gauss_strng[3,:,:] ,levels=0.5*np.linspace(-1000,1000,num=31) , cmap='terrain' ) #np.asarray( [6,7.5,9,12,18,25,37,50,75] ) )
ax.scatter( ixs, iys, marker='+', color='black' )
ax.set_xlim( xlim )
ax.set_ylim( ylim )
plt.colorbar(co)



In [ ]:
import matplotlib as mpl

levels = np.linspace(0, 80, num=31)                     # same levels as your contourf
cmap   = plt.get_cmap('viridis')
norm   = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, clip=True)

jx = np.asarray(ixs).astype(int)                        # seed indices as you plotted them
jy = np.asarray(iys).astype(int)
s_at_seed = gauss_scale[3, jy, jx]                      # value at each seed
inbox = (jx >= xlim[0]) & (jx <= xlim[1]) & (jy >= ylim[0]) & (jy <= ylim[1])
ridge = inbox & (s_at_seed > 0)
none  = inbox & (s_at_seed == 0)

fig, axs = plt.subplots(1, 2, figsize=(20, 8))

ax = axs[0]
ax.set_facecolor('0.85')                                # light grey, so dark dots show
sc = ax.scatter(jx[ridge], jy[ridge], c=s_at_seed[ridge], cmap=cmap, norm=norm,
                marker='o', s=20, edgecolors='none')
ax.scatter(jx[none], jy[none], marker='x', s=12, color='red', label='no ridge scale (0)')
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
ax.legend(loc='lower right')
plt.colorbar(sc, ax=ax, label='gauss_scale at seed (km)')

ax = axs[1]
co = ax.contourf(dev[3, :, :], levels=np.linspace(-1000, 1000, num=31), cmap='terrain')
ax.scatter(ixs, iys, marker='+', color='black')
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
plt.colorbar(co, ax=ax)

In [ ]:
sig=np.asarray( [6,7.5,9,12,18,25,37,50,75] )



# --- sanity: blurrier snapshots move away from raw and toward terr_sm -----
#for k in range(len(sig)):
#    print(f"{sig[k]:6.1f} km   rms(snap-raw) {np.std(S[k]-(sm+dev)):7.1f} m   rms(snap-terr_sm) {np.std(S[k]-sm):7.1f} m")



"""
# --- region: Tennessee on face 4 (same as before) -------------------------
p = 3
m = CP.in_box(tlons[p], tlats[p], [268., 285.], [30., 41.])
yy, xx = np.nonzero(m)
y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1
"""
# --- region: SW US on face 4 (same as before) -------------------------
p = 3
m = CP.in_box(tlons[p], tlats[p], [268.-30, 285.-30], [30., 41.])
yy, xx = np.nonzero(m)
y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1

# --- region: Greenland on face 6 (same as before) -------------------------
p = 5
m = CP.in_box(tlons[p], tlats[p], [-80+360.,-20+360.], [58., 70.])
yy, xx = np.nonzero(m)
y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1


# (b) the same widths with gaussian_filter on dev (crop with a margin, filter, then trim)
pad = int(4 * sig.max() / 3.336) + 1
Y0, Y1, X0, X1 = max(y0-pad, 0), min(y1+pad, nc), max(x0-pad, 0), min(x1+pad, nc)
D = dev[p, Y0:Y1, X0:X1]
fields_gauss = [gaussian_filter(D, s/3.336)[y0-Y0:y1-Y0, x0-X0:x1-X0] for s in sig]


print( np.shape( D ) )

sB, rB, tB = ridge_scale_from_fields(fields_gauss, sig )
print( np.shape( sB ) )


sA, rA, tA = gauss_scale[p, y0:y1, x0:x1] , gauss_strng[p, y0:y1, x0:x1] , gauss_angle[p, y0:y1, x0:x1] 
print( np.shape( sA) )

In [ ]:
# --- side by side, clear ridges only (R1 ~ 0.35 x ridge height) ------------
fig, ax = plt.subplots(1, 2, figsize=(18, 8))
#for a, s_, r_, title in [(ax[0], sA, rA, 'smoother snapshots - terr_sm'),
for a, s_, r_, title in [(ax[0], sA, rA, 'F90 index-space filter'),
                         (ax[1], sB, rB, 'Python gaussian_filter(dev)')]:
    im = a.imshow(np.where(r_ > 50., s_, np.nan), origin='lower', cmap='viridis',
                  vmin=sig.min(), vmax=sig.max(), interpolation='nearest')
    a.set_title(title); plt.colorbar(im, ax=a, shrink=0.6, label='best sigma (km)')
#both = (rA > 50.) & (rB > 50.)
#ax[2].hist2d(sB[both], sA[both], bins=[np.append(sig, sig[-1]*1.5)]*2, cmap='Greys')
#ax[2].set_xlabel('gaussian_filter(dev): best sigma'); ax[2].set_ylabel('snapshots: best sigma')
#ax[2].set_title('agreement (diagonal = same scale)')
#print("same scale at", np.mean(sA[both] == sB[both]).round(2), "of clear-ridge pixels")